# 🧠 Motor Biométrico Facial - InsightFace (ArcFace)
### Sistema de Autenticación Facial Offline
- ✅ Registro biométrico en tiempo real con guía visual (óvalo)
- 🧬 Generación de embeddings faciales (512D)
- 🔐 Verificación de identidad 1:1
- 🛡️ Detección básica de spoofing (parpadeo + movimiento)
- 📊 Validación de calidad de imagen

## 📦 Bloque 1: Imports y Configuración
Importamos todas las librerías necesarias y definimos las constantes del sistema (umbrales de calidad, similaridad, rutas de almacenamiento).

In [42]:
import cv2
import numpy as np
import insightface
from insightface.app import FaceAnalysis
import os
import json
import time
from datetime import datetime
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# ===================== CONFIGURACIÓN =====================
# Rutas
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
PROFILES_DIR = os.path.join(BASE_DIR, "profiles")
os.makedirs(PROFILES_DIR, exist_ok=True)

# Umbrales de calidad
BLUR_THRESHOLD = 80.0          # Varianza Laplaciana mínima (nitidez)
BRIGHTNESS_MIN = 50            # Brillo mínimo (0-255)
BRIGHTNESS_MAX = 200           # Brillo máximo (0-255)
MIN_FACE_RATIO = 0.08          # Tamaño mínimo del rostro vs frame (8%)
CENTER_TOLERANCE = 0.25        # Tolerancia de centrado (25% del frame)

# Umbrales de similaridad (cosine similarity)
MATCH_THRESHOLD = 0.45         # Score mínimo para match (cosine distance)
GRAY_ZONE_MIN = 0.35           # Zona gris inferior
MAX_LOGIN_ATTEMPTS = 3         # Intentos máximos antes de bloqueo

# Registro
MIN_CAPTURES = 8               # Capturas mínimas para registro
MAX_CAPTURES = 15              # Capturas máximas
REGISTER_TIMEOUT = 12          # Segundos máximos de registro
CAPTURE_INTERVAL = 0.4         # Segundos entre capturas válidas

# Cámara
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480

# Colores BGR
GREEN = (0, 255, 0)
RED = (0, 0, 255)
YELLOW = (0, 255, 255)
WHITE = (255, 255, 255)
CYAN = (255, 255, 0)
ORANGE = (0, 165, 255)

print("✅ Configuración cargada")
print(f"📁 Perfiles en: {PROFILES_DIR}")

✅ Configuración cargada
📁 Perfiles en: /home/andi/vXcode/Lofasys/model_insightface/profiles


## 🤖 Bloque 2: Cargar Modelo InsightFace
Inicializamos el modelo **buffalo_l** (ArcFace) para detección y reconocimiento facial. El modelo se descarga automáticamente la primera vez (~300MB). Funciona en CPU.

In [43]:
print("🔄 Cargando modelo InsightFace (buffalo_l)...")
start = time.time()

# Inicializar FaceAnalysis con modelo buffalo_l
face_app = FaceAnalysis(
    name='buffalo_l',
    root=os.path.join(BASE_DIR, "models"),
    providers=['CPUExecutionProvider']
)

# Preparar para resolución de cámara
face_app.prepare(ctx_id=0, det_size=(640, 640))

elapsed = time.time() - start
print(f"✅ Modelo cargado en {elapsed:.1f}s")
print(f"🧠 Modelos disponibles: {list(face_app.models.keys()) if isinstance(face_app.models, dict) else [str(m) for m in face_app.models]}")

🔄 Cargando modelo InsightFace (buffalo_l)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'

## ✅ Bloque 3: Funciones de Validación de Calidad
Validaciones antes de aceptar una captura:
- **Nitidez**: Varianza Laplaciana (detecta imágenes borrosas)
- **Iluminación**: Brillo promedio del rostro (ni muy oscuro ni sobreexpuesto)
- **Tamaño del rostro**: Debe ocupar al menos 8% del frame
- **Centrado**: El rostro debe estar dentro de la zona guía (óvalo)
- **Rostro único**: Solo debe haber un rostro en la escena

In [44]:
def check_blur(face_img):
    """Verifica nitidez usando varianza Laplaciana."""
    gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var >= BLUR_THRESHOLD, laplacian_var

def check_brightness(face_img):
    """Verifica iluminación adecuada del rostro."""
    gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    mean_brightness = np.mean(gray)
    ok = BRIGHTNESS_MIN <= mean_brightness <= BRIGHTNESS_MAX
    return ok, mean_brightness

def check_face_size(bbox, frame_shape):
    """Verifica que el rostro sea suficientemente grande."""
    h, w = frame_shape[:2]
    face_w = bbox[2] - bbox[0]
    face_h = bbox[3] - bbox[1]
    face_area = face_w * face_h
    frame_area = w * h
    ratio = face_area / frame_area
    return ratio >= MIN_FACE_RATIO, ratio

def check_face_centered(bbox, frame_shape):
    """Verifica que el rostro esté centrado en el frame."""
    h, w = frame_shape[:2]
    face_cx = (bbox[0] + bbox[2]) / 2
    face_cy = (bbox[1] + bbox[3]) / 2
    frame_cx, frame_cy = w / 2, h / 2

    dx = abs(face_cx - frame_cx) / w
    dy = abs(face_cy - frame_cy) / h

    centered = dx < CENTER_TOLERANCE and dy < CENTER_TOLERANCE
    return centered, (dx, dy)

def validate_face(face, frame):
    """Ejecuta todas las validaciones y retorna (ok, mensajes)."""
    messages = []
    all_ok = True
    bbox = face.bbox.astype(int)

    # Extraer región del rostro
    x1, y1, x2, y2 = bbox
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
    face_img = frame[y1:y2, x1:x2]

    if face_img.size == 0:
        return False, ["❌ Rostro fuera de cuadro"]

    # 1. Nitidez
    ok, val = check_blur(face_img)
    if not ok:
        messages.append(f"🌫️ Imagen borrosa ({val:.0f})")
        all_ok = False

    # 2. Iluminación
    ok, val = check_brightness(face_img)
    if not ok:
        msg = "oscuro" if val < BRIGHTNESS_MIN else "sobreexpuesto"
        messages.append(f"💡 Muy {msg} ({val:.0f})")
        all_ok = False

    # 3. Tamaño
    ok, val = check_face_size(bbox, frame.shape)
    if not ok:
        messages.append(f"📏 Acérquese más ({val*100:.1f}%)")
        all_ok = False

    # 4. Centrado
    ok, (dx, dy) = check_face_centered(bbox, frame.shape)
    if not ok:
        messages.append("🎯 Centre su rostro en el óvalo")
        all_ok = False

    if all_ok:
        messages.append("✅ Calidad OK")

    return all_ok, messages

print("✅ Funciones de validación de calidad cargadas")

✅ Funciones de validación de calidad cargadas


## 🛡️ Bloque 4: Detección de Liveness (Anti-Spoofing)
Detecta si hay una persona real frente a la cámara:
- **Parpadeo**: Usa los landmarks de los ojos (EAR - Eye Aspect Ratio). Si el ratio baja, hay parpadeo.
- **Movimiento de cabeza**: Compara la posición del rostro entre frames para detectar movimiento natural.
- Si no parpadeas o te quedas completamente quieto → posible foto/pantalla.

In [45]:
class LivenessDetector:
    """Detector de liveness usando parpadeo y movimiento de cabeza."""

    def __init__(self):
        self.blink_count = 0
        self.prev_ear = None
        self.ear_threshold = 0.22       # Umbral EAR para detectar ojo cerrado
        self.blink_consec_frames = 2
        self.blink_counter = 0
        self.prev_nose_pos = None
        self.movement_detected = False
        self.movement_threshold = 8.0   # Píxeles mínimos de movimiento

    def eye_aspect_ratio(self, landmarks):
        """Calcula EAR usando los 5 landmarks de InsightFace.
        landmarks: array (5,2) -> [ojo_izq, ojo_der, nariz, boca_izq, boca_der]
        Usamos distancia entre ojos y boca como proxy de EAR.
        """
        left_eye = landmarks[0]   # Ojo izquierdo
        right_eye = landmarks[1]  # Ojo derecho
        nose = landmarks[2]       # Nariz

        # Distancia vertical: ojos a nariz
        eye_dist = np.linalg.norm(left_eye - right_eye)
        left_nose = np.linalg.norm(left_eye - nose)
        right_nose = np.linalg.norm(right_eye - nose)

        # Ratio: si los ojos se cierran, la distancia cambia
        ear = (left_nose + right_nose) / (2.0 * eye_dist + 1e-6)
        return ear

    def detect_blink(self, landmarks):
        """Detecta parpadeo comparando EAR entre frames."""
        ear = self.eye_aspect_ratio(landmarks)

        if self.prev_ear is not None:
            # Si EAR baja significativamente → parpadeo
            ear_diff = self.prev_ear - ear
            if ear_diff > 0.02:
                self.blink_counter += 1
            else:
                if self.blink_counter >= self.blink_consec_frames:
                    self.blink_count += 1
                self.blink_counter = 0

        self.prev_ear = ear
        return self.blink_count > 0

    def detect_movement(self, landmarks):
        """Detecta movimiento natural de cabeza usando posición de nariz."""
        nose = landmarks[2]

        if self.prev_nose_pos is not None:
            dist = np.linalg.norm(nose - self.prev_nose_pos)
            if dist > self.movement_threshold:
                self.movement_detected = True

        self.prev_nose_pos = nose.copy()
        return self.movement_detected

    def check_liveness(self, landmarks):
        """Ejecuta todas las verificaciones de liveness."""
        blink_ok = self.detect_blink(landmarks)
        movement_ok = self.detect_movement(landmarks)

        # Para registro: necesitamos al menos movimiento O parpadeo
        is_alive = blink_ok or movement_ok

        status = []
        status.append(f"👁️ Parpadeo: {'✅' if blink_ok else '⏳ parpadee'}")
        status.append(f"↔️ Movimiento: {'✅' if movement_ok else '⏳ mueva la cabeza'}")

        return is_alive, status

    def reset(self):
        """Reinicia el detector para nueva sesión."""
        self.blink_count = 0
        self.prev_ear = None
        self.blink_counter = 0
        self.prev_nose_pos = None
        self.movement_detected = False

print("✅ Detector de Liveness cargado")

✅ Detector de Liveness cargado


## 🧬 Bloque 5: Gestión de Embeddings
Funciones para:
- **Generar embedding promedio** a partir de múltiples capturas (más robusto que una sola)
- **Guardar perfil** en archivo `.npz` (embedding + metadata)
- **Cargar perfil** existente para comparación
- **Comparar embeddings** usando distancia coseno (score 0.0 - 1.0)
- **Actualización adaptativa** del perfil tras login exitoso: $E_{nuevo} = \frac{E_{perfil} \cdot k + E_{actual}}{k + 1}$

In [46]:
def compute_average_embedding(embeddings):
    """Calcula el embedding promedio normalizado de una lista de embeddings."""
    if not embeddings:
        return None
    avg = np.mean(embeddings, axis=0)
    # Normalizar L2
    norm = np.linalg.norm(avg)
    if norm > 0:
        avg = avg / norm
    return avg

def cosine_similarity(emb1, emb2):
    """Calcula similaridad coseno entre dos embeddings."""
    dot = np.dot(emb1, emb2)
    norm1 = np.linalg.norm(emb1)
    norm2 = np.linalg.norm(emb2)
    return dot / (norm1 * norm2 + 1e-6)

def cosine_distance(emb1, emb2):
    """Distancia coseno (menor = más similar). InsightFace usa esto."""
    return 1.0 - cosine_similarity(emb1, emb2)

def save_profile(username, embedding, num_captures):
    """Guarda un perfil biométrico en disco."""
    filepath = os.path.join(PROFILES_DIR, f"{username}.npz")
    metadata = {
        "username": username,
        "created": datetime.now().isoformat(),
        "num_captures": int(num_captures),
        "login_count": 0,
        "update_weight": 5  # k inicial para actualización adaptativa
    }
    np.savez(filepath,
             embedding=embedding,
             metadata=json.dumps(metadata))
    print(f"💾 Perfil guardado: {filepath}")
    return filepath

def load_profile(username):
    """Carga un perfil biométrico existente."""
    filepath = os.path.join(PROFILES_DIR, f"{username}.npz")
    if not os.path.exists(filepath):
        return None, None
    data = np.load(filepath, allow_pickle=True)
    embedding = data['embedding']
    metadata = json.loads(str(data['metadata']))
    return embedding, metadata

def update_profile_adaptive(username, new_embedding):
    """Actualiza el perfil tras login exitoso (aprendizaje adaptativo).
    E_nuevo = (E_perfil * k + E_actual) / (k + 1)
    """
    embedding, metadata = load_profile(username)
    if embedding is None:
        return False

    k = metadata.get('update_weight', 5)
    updated = (embedding * k + new_embedding) / (k + 1)
    # Normalizar
    norm = np.linalg.norm(updated)
    if norm > 0:
        updated = updated / norm

    metadata['login_count'] = metadata.get('login_count', 0) + 1
    metadata['last_login'] = datetime.now().isoformat()
    metadata['update_weight'] = min(k + 1, 20)  # Tope en 20

    filepath = os.path.join(PROFILES_DIR, f"{username}.npz")
    np.savez(filepath,
             embedding=updated,
             metadata=json.dumps(metadata))
    return True

def list_profiles():
    """Lista todos los perfiles registrados."""
    profiles = [f.replace('.npz', '') for f in os.listdir(PROFILES_DIR) if f.endswith('.npz')]
    return profiles

print("✅ Gestión de embeddings cargada")
print(f"👤 Perfiles existentes: {list_profiles()}")

✅ Gestión de embeddings cargada
👤 Perfiles existentes: []


## 🎨 Bloque 6: Dibujo de Interfaz Visual (Óvalo Guía + HUD)
Funciones de dibujo en el video:
- **Óvalo guía**: Muestra dónde colocar la cara (cambia de color: rojo → amarillo → verde)
- **Bounding box** del rostro detectado
- **Panel de estado** con mensajes de calidad y liveness
- **Barra de progreso** de capturas durante el registro
- **Score de similaridad** durante el login

In [48]:
def draw_face_guide(frame, color=WHITE, thickness=2):
    """Dibuja un óvalo guía centrado donde el usuario debe colocar su rostro."""
    h, w = frame.shape[:2]
    center = (w // 2, h // 2 - 20)
    # Óvalo proporcional al frame
    axes = (int(w * 0.22), int(h * 0.35))
    cv2.ellipse(frame, center, axes, 0, 0, 360, color, thickness)

    # Líneas de referencia (cruz central sutil)
    line_len = 15
    cx, cy = center
    cv2.line(frame, (cx - line_len, cy), (cx + line_len, cy), color, 1)
    cv2.line(frame, (cx, cy - line_len), (cx, cy + line_len), color, 1)

    return center, axes

def draw_face_bbox(frame, face, color=GREEN):
    """Dibuja el bounding box del rostro detectado."""
    bbox = face.bbox.astype(int)
    x1, y1, x2, y2 = bbox

    # Esquinas estilizadas (no un rectángulo completo)
    corner_len = 20
    t = 2

    # Esquina superior izquierda
    cv2.line(frame, (x1, y1), (x1 + corner_len, y1), color, t)
    cv2.line(frame, (x1, y1), (x1, y1 + corner_len), color, t)
    # Esquina superior derecha
    cv2.line(frame, (x2, y1), (x2 - corner_len, y1), color, t)
    cv2.line(frame, (x2, y1), (x2, y1 + corner_len), color, t)
    # Esquina inferior izquierda
    cv2.line(frame, (x1, y2), (x1 + corner_len, y2), color, t)
    cv2.line(frame, (x1, y2), (x1, y2 - corner_len), color, t)
    # Esquina inferior derecha
    cv2.line(frame, (x2, y2), (x2 - corner_len, y2), color, t)
    cv2.line(frame, (x2, y2), (x2, y2 - corner_len), color, t)

    # Score de detección
    if hasattr(face, 'det_score'):
        cv2.putText(frame, f"{face.det_score:.2f}",
                    (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

def draw_status_panel(frame, messages, y_start=30):
    """Dibuja panel de estado con mensajes en la parte superior."""
    # Fondo semitransparente
    overlay = frame.copy()
    panel_h = 25 + len(messages) * 25
    cv2.rectangle(overlay, (5, 5), (380, panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    y = y_start
    for msg in messages:
        color = GREEN if "✅" in msg else (YELLOW if "⏳" in msg else RED)
        cv2.putText(frame, msg, (15, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        y += 25

def draw_progress_bar(frame, current, total, y_pos=None):
    """Dibuja barra de progreso de capturas."""
    h, w = frame.shape[:2]
    if y_pos is None:
        y_pos = h - 40

    bar_w = int(w * 0.6)
    bar_x = (w - bar_w) // 2
    bar_h = 20

    # Fondo
    cv2.rectangle(frame, (bar_x, y_pos), (bar_x + bar_w, y_pos + bar_h), (50, 50, 50), -1)
    # Progreso
    progress = min(current / total, 1.0)
    fill_w = int(bar_w * progress)
    color = GREEN if progress >= 1.0 else CYAN
    cv2.rectangle(frame, (bar_x, y_pos), (bar_x + fill_w, y_pos + bar_h), color, -1)
    # Texto
    text = f"Capturas: {current}/{total}"
    cv2.putText(frame, text, (bar_x + 5, y_pos + 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1, cv2.LINE_AA)
    # Borde
    cv2.rectangle(frame, (bar_x, y_pos), (bar_x + bar_w, y_pos + bar_h), WHITE, 1)

def draw_countdown(frame, seconds_left):
    """Dibuja cuenta regresiva."""
    h, w = frame.shape[:2]
    text = f"{seconds_left:.0f}s"
    cv2.putText(frame, text, (w - 70, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, YELLOW, 2, cv2.LINE_AA)

def draw_result_overlay(frame, text, color=GREEN):
    """Dibuja un resultado grande centrado."""
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, h//2 - 50), (w, h//2 + 50), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

    font_scale = 1.2
    thickness = 3
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    x = (w - tw) // 2
    y = h // 2 + th // 2
    cv2.putText(frame, text, (x, y),
                cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness, cv2.LINE_AA)

print("✅ Funciones de interfaz visual cargadas")

✅ Funciones de interfaz visual cargadas


## 📷 Bloque 7: Registro Facial (Crear Perfil)
Proceso completo de registro:
1. Abre la cámara con el **óvalo guía** donde debes colocar tu cara
2. Detecta tu rostro y valida calidad en tiempo real
3. Verifica liveness (parpadeo / movimiento)
4. Captura **8-15 frames válidos** durante ~12 segundos
5. Calcula el **embedding promedio** (512D) y guarda el perfil

> El óvalo cambia de color: 🔴 Rojo (no detectado) → 🟡 Amarillo (detectado, validando) → 🟢 Verde (captura válida)

In [50]:
def register_face(username):
    """Proceso completo de registro facial con video, óvalo guía y feedback visual."""
    print(f"📷 Iniciando registro para: {username}")
    print("🎯 Coloque su rostro dentro del óvalo. Parpadee y mueva ligeramente la cabeza.")
    print("⌨️  Presione 'q' para cancelar.\n")

    # Verificar si ya existe
    existing, _ = load_profile(username)
    if existing is not None:
        print(f"⚠️  Ya existe un perfil para '{username}'. Se sobreescribirá.")

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)

    if not cap.isOpened():
        print("❌ Error: No se pudo abrir la cámara")
        return False

    liveness = LivenessDetector()
    embeddings = []
    last_capture_time = 0
    start_time = time.time()
    frame_count = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)  # Espejo
            elapsed = time.time() - start_time
            remaining = REGISTER_TIMEOUT - elapsed

            # Timeout
            if remaining <= 0:
                if len(embeddings) >= MIN_CAPTURES:
                    break
                else:
                    draw_result_overlay(frame, "TIEMPO AGOTADO - Pocas capturas", RED)
                    cv2.imshow("Registro Facial", frame)
                    cv2.waitKey(2000)
                    print(f"❌ Tiempo agotado. Solo {len(embeddings)}/{MIN_CAPTURES} capturas.")
                    return False

            # Suficientes capturas
            if len(embeddings) >= MAX_CAPTURES:
                break

            # Detectar rostros
            faces = face_app.get(frame)
            guide_color = RED
            status_msgs = [f"👤 Registro: {username}"]

            if len(faces) == 0:
                status_msgs.append("❌ No se detecta rostro")
            elif len(faces) > 1:
                status_msgs.append("❌ Múltiples rostros detectados")
                guide_color = RED
            else:
                face = faces[0]
                # Validar calidad
                quality_ok, quality_msgs = validate_face(face, frame)
                status_msgs.extend(quality_msgs)

                # Dibujar bbox
                bbox_color = YELLOW if not quality_ok else GREEN
                draw_face_bbox(frame, face, bbox_color)

                if quality_ok:
                    guide_color = YELLOW
                    # Verificar liveness
                    if face.kps is not None:
                        alive, live_msgs = liveness.check_liveness(face.kps)
                        status_msgs.extend(live_msgs)

                        if alive:
                            guide_color = GREEN
                            # Capturar embedding con intervalo
                            now = time.time()
                            if now - last_capture_time >= CAPTURE_INTERVAL:
                                if face.embedding is not None:
                                    emb = face.embedding / (np.linalg.norm(face.embedding) + 1e-6)
                                    embeddings.append(emb)
                                    last_capture_time = now
                                    status_msgs.append(f"📸 Captura #{len(embeddings)}")
                    else:
                        status_msgs.append("⏳ Verificando liveness...")
                        guide_color = YELLOW

            # Dibujar interfaz
            draw_face_guide(frame, guide_color, 2)
            draw_status_panel(frame, status_msgs)
            draw_progress_bar(frame, len(embeddings), MIN_CAPTURES)
            draw_countdown(frame, max(0, remaining))

            cv2.imshow("Registro Facial", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("⚠️  Registro cancelado por el usuario.")
                return False

    finally:
        cap.release()
        cv2.destroyAllWindows()

    # Calcular embedding promedio
    if len(embeddings) < MIN_CAPTURES:
        print(f"❌ Capturas insuficientes: {len(embeddings)}/{MIN_CAPTURES}")
        return False

    avg_embedding = compute_average_embedding(embeddings)
    save_profile(username, avg_embedding, len(embeddings))

    print(f"\n✅ ¡Registro exitoso!")
    print(f"📊 Capturas válidas: {len(embeddings)}")
    print(f"🧬 Embedding: vector de {len(avg_embedding)} dimensiones")
    print(f"⏱️  Tiempo: {time.time() - start_time:.1f}s")
    return True

print("✅ Función de registro facial cargada")

✅ Función de registro facial cargada


## 🔐 Bloque 8: Login Facial (Verificación 1:1)
Proceso de login:
1. Abre la cámara con el **óvalo guía**
2. Detecta y valida el rostro
3. Genera embedding y lo **compara con el perfil guardado**
4. Muestra **score de similaridad** en tiempo real
5. Resultado: ✅ Match (>0.45) | 🔄 Zona gris (0.35-0.45) | ❌ No match (<0.35)
6. Si hay match → **actualiza el perfil adaptativamente**
7. Tras 3 fallos → sugiere login con contraseña

In [51]:
def login_face(username):
    """Proceso de login facial con verificación 1:1."""
    print(f"🔐 Iniciando login para: {username}")

    # Cargar perfil
    profile_emb, metadata = load_profile(username)
    if profile_emb is None:
        print(f"❌ No existe perfil para '{username}'. Regístrese primero.")
        return None

    print(f"📋 Perfil cargado - Logins previos: {metadata.get('login_count', 0)}")
    print("🎯 Coloque su rostro dentro del óvalo.")
    print("⌨️  Presione 'q' para cancelar.\n")

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)

    if not cap.isOpened():
        print("❌ Error: No se pudo abrir la cámara")
        return None

    liveness = LivenessDetector()
    attempts = 0
    best_score = 0
    start_time = time.time()
    login_timeout = 15  # 15 segundos para login
    result = None
    verified_frames = 0
    required_verified = 3  # Necesita 3 frames verificados consecutivos

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            elapsed = time.time() - start_time
            remaining = login_timeout - elapsed

            if remaining <= 0:
                draw_result_overlay(frame, "TIEMPO AGOTADO", RED)
                cv2.imshow("Login Facial", frame)
                cv2.waitKey(2000)
                result = {
                    "match": False, "score": best_score,
                    "liveness": False, "quality_ok": False,
                    "message": "Tiempo agotado"
                }
                break

            faces = face_app.get(frame)
            guide_color = RED
            status_msgs = [f"🔐 Login: {username} | Intento {attempts + 1}/{MAX_LOGIN_ATTEMPTS}"]
            current_score = 0

            if len(faces) == 0:
                status_msgs.append("❌ No se detecta rostro")
                verified_frames = 0
            elif len(faces) > 1:
                status_msgs.append("❌ Múltiples rostros")
                verified_frames = 0
            else:
                face = faces[0]
                quality_ok, quality_msgs = validate_face(face, frame)
                status_msgs.extend(quality_msgs)
                draw_face_bbox(frame, face, YELLOW)

                if quality_ok and face.embedding is not None:
                    guide_color = YELLOW

                    # Verificar liveness
                    alive = True  # En login, liveness es más permisivo
                    if face.kps is not None:
                        alive, live_msgs = liveness.check_liveness(face.kps)
                        status_msgs.extend(live_msgs)

                    # Calcular similaridad
                    emb = face.embedding / (np.linalg.norm(face.embedding) + 1e-6)
                    dist = cosine_distance(emb, profile_emb)
                    sim = cosine_similarity(emb, profile_emb)
                    current_score = max(sim, 0)
                    best_score = max(best_score, current_score)

                    # Mostrar score
                    score_color = GREEN if dist < MATCH_THRESHOLD else (YELLOW if dist < GRAY_ZONE_MIN else RED)
                    status_msgs.append(f"📊 Similaridad: {current_score:.3f} (dist: {dist:.3f})")

                    # Evaluar match
                    if dist < MATCH_THRESHOLD:
                        guide_color = GREEN
                        verified_frames += 1
                        status_msgs.append(f"✅ Match! ({verified_frames}/{required_verified})")

                        if verified_frames >= required_verified:
                            # ¡LOGIN EXITOSO!
                            draw_face_guide(frame, GREEN, 3)
                            draw_face_bbox(frame, face, GREEN)
                            draw_status_panel(frame, status_msgs)
                            draw_result_overlay(frame, f"ACCESO CONCEDIDO - {current_score:.2f}", GREEN)
                            cv2.imshow("Login Facial", frame)
                            cv2.waitKey(2000)

                            # Actualizar perfil adaptativamente
                            update_profile_adaptive(username, emb)

                            result = {
                                "match": True,
                                "score": float(current_score),
                                "liveness": alive,
                                "quality_ok": True,
                                "timestamp": datetime.now().isoformat(),
                                "processing_time_ms": int((time.time() - start_time) * 1000),
                                "confidence_level": "high" if current_score > 0.6 else "medium"
                            }
                            break
                    elif dist < GRAY_ZONE_MIN + 0.15:
                        guide_color = YELLOW
                        status_msgs.append("🔄 Zona gris - Ajuste posición")
                        verified_frames = 0
                    else:
                        guide_color = RED
                        verified_frames = 0
                        status_msgs.append("❌ No coincide")

                    # Score visual en la parte inferior
                    h, w = frame.shape[:2]
                    bar_w = int(w * 0.5)
                    bar_x = (w - bar_w) // 2
                    bar_y = h - 70
                    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + 20), (50, 50, 50), -1)
                    fill = int(bar_w * min(current_score, 1.0))
                    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + fill, bar_y + 20), score_color, -1)
                    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + 20), WHITE, 1)
                    # Marca del umbral
                    thresh_x = bar_x + int(bar_w * (1 - MATCH_THRESHOLD))
                    cv2.line(frame, (thresh_x, bar_y - 5), (thresh_x, bar_y + 25), WHITE, 2)
                    cv2.putText(frame, f"Score: {current_score:.3f}",
                                (bar_x, bar_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1)

            # Dibujar interfaz
            draw_face_guide(frame, guide_color, 2)
            draw_status_panel(frame, status_msgs)
            draw_countdown(frame, max(0, remaining))

            cv2.imshow("Login Facial", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                result = {"match": False, "score": best_score, "message": "Cancelado"}
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

    if result is None:
        result = {
            "match": False, "score": float(best_score),
            "liveness": False, "quality_ok": False,
            "message": "Sin resultado"
        }

    # Imprimir resultado
    print("\n" + "=" * 50)
    if result["match"]:
        print(f"✅ LOGIN EXITOSO")
        print(f"📊 Score: {result['score']:.3f}")
        print(f"⏱️  Tiempo: {result.get('processing_time_ms', 0)}ms")
        print(f"🔒 Confianza: {result.get('confidence_level', 'N/A')}")
    else:
        print(f"❌ LOGIN FALLIDO")
        print(f"📊 Mejor score: {result['score']:.3f}")
        print(f"💬 {result.get('message', 'No match')}")
    print("=" * 50)

    return result

print("✅ Función de login facial cargada")

✅ Función de login facial cargada


## 🚀 Bloque 9: Ejecutar - Registro
Ejecuta esta celda para **registrar tu cara**. Se lanza el script externo `face_engine.py` que abre la ventana de cámara con el óvalo guía.

**Instrucciones:**
1. Coloca tu cara dentro del óvalo
2. Parpadea normalmente y mueve ligeramente la cabeza
3. Espera a que se completen las capturas (barra verde)
4. Presiona `q` para cancelar

> ⚠️ Se ejecuta como proceso externo para tener acceso completo a GUI (cv2.imshow).

In [54]:
# =============================================
# 📷 REGISTRO: Cambia el nombre de usuario aquí
# =============================================
import subprocess, json, sys, shutil

USERNAME = "andi"

# Usar el Python del env conda 'insightface' (tiene opencv con GUI)
python_exe = shutil.which("python", path=os.path.expanduser("~/anaconda3/envs/insightface/bin"))
if not python_exe:
    python_exe = sys.executable  # fallback
engine_path = os.path.join(BASE_DIR, "face_engine.py")

print(f"🚀 Lanzando registro facial para '{USERNAME}'...")
print(f"🐍 Python: {python_exe}")
print("📺 Se abrirá una ventana de cámara. Coloca tu cara en el óvalo.\n")

proc = subprocess.run(
    [python_exe, engine_path, "register", USERNAME],
    capture_output=True, text=True, cwd=BASE_DIR
)

# Mostrar output del proceso
result = None
if proc.stdout:
    for line in proc.stdout.strip().split('\n'):
        if line.startswith("__RESULT_JSON__:"):
            result = json.loads(line.replace("__RESULT_JSON__:", ""))
        else:
            print(line)

print("\n" + "=" * 50)
if result:
    if result.get("success"):
        print(f"✅ ¡Registro exitoso!")
        print(f"👤 Usuario: {result['username']}")
        print(f"📸 Capturas: {result['num_captures']}")
        print(f"🧬 Embedding: {result['embedding_dim']}D")
        print(f"⏱️  Tiempo: {result['processing_time_s']}s")
        print(f"💾 Perfil: {result['profile_path']}")
        print("\n🎉 Ahora puedes usar el login facial en el siguiente bloque.")
    else:
        print(f"⚠️  Registro fallido: {result.get('message', 'Error desconocido')}")
else:
    print("❌ No se obtuvo resultado del proceso")
    if proc.stderr:
        errors = [l for l in proc.stderr.split('\n') if l and 'QThread' not in l]
        for e in errors[:10]:
            print(f"  {e}")
print("=" * 50)

🚀 Lanzando registro facial para 'andi'...
🐍 Python: /home/andi/anaconda3/envs/insightface/bin/python
📺 Se abrirá una ventana de cámara. Coloca tu cara en el óvalo.

🔄 Cargando modelo InsightFace (buffalo_l)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode

## 🔐 Bloque 10: Ejecutar - Login
Ejecuta esta celda para **iniciar sesión con tu cara**. Se lanza `face_engine.py` con la acción login.

**Resultado:**
- ✅ Score > 0.45 → Acceso concedido (verde)
- 🔄 Score 0.35-0.45 → Zona gris, ajuste posición (amarillo)
- ❌ Score < 0.35 → Acceso denegado (rojo)

In [55]:
# =============================================
# 🔐 LOGIN: Cambia el nombre de usuario aquí
# =============================================
import subprocess, json, sys, shutil

USERNAME = "andi"

python_exe = shutil.which("python", path=os.path.expanduser("~/anaconda3/envs/insightface/bin"))
if not python_exe:
    python_exe = sys.executable
engine_path = os.path.join(BASE_DIR, "face_engine.py")

print(f"🚀 Lanzando login facial para '{USERNAME}'...")
print("📺 Se abrirá una ventana de cámara. Coloca tu cara en el óvalo.\n")

proc = subprocess.run(
    [python_exe, engine_path, "login", USERNAME],
    capture_output=True, text=True, cwd=BASE_DIR
)

result = None
if proc.stdout:
    for line in proc.stdout.strip().split('\n'):
        if line.startswith("__RESULT_JSON__:"):
            result = json.loads(line.replace("__RESULT_JSON__:", ""))
        else:
            print(line)

print("\n" + "=" * 50)
if result:
    if result.get("match"):
        print(f"✅ LOGIN EXITOSO")
        print(f"📊 Score: {result['score']:.3f}")
        print(f"⏱️  Tiempo: {result.get('processing_time_ms', 0)}ms")
        print(f"🔒 Confianza: {result.get('confidence_level', 'N/A')}")
    else:
        print(f"❌ LOGIN FALLIDO")
        print(f"📊 Mejor score: {result.get('score', 0):.3f}")
        print(f"💬 {result.get('message', 'No match')}")

    print("=" * 50)
    print("\n📤 Resultado (JSON):")
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("❌ No se obtuvo resultado del proceso")
    if proc.stderr:
        errors = [l for l in proc.stderr.split('\n') if l and 'QThread' not in l]
        for e in errors[:10]:
            print(f"  {e}")
    print("=" * 50)

🚀 Lanzando login facial para 'andi'...
📺 Se abrirá una ventana de cámara. Coloca tu cara en el óvalo.

🔄 Cargando modelo InsightFace (buffalo_l)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/andi/vXcode/Lofasys/model_insightface/models/models/buffalo_l/genderage.o

## 🛠️ Bloque 11: Utilidades
Funciones extra para gestionar perfiles:
- Ver perfiles registrados
- Eliminar un perfil
- Ver información de un perfil

In [61]:
# =============================================
# 🛠️ UTILIDADES (via face_engine.py)
# =============================================
import subprocess, json, sys, shutil

python_exe = shutil.which("python", path=os.path.expanduser("~/anaconda3/envs/insightface/bin"))
if not python_exe:
    python_exe = sys.executable
engine_path = os.path.join(BASE_DIR, "face_engine.py")

def run_engine(action, username=None):
    """Ejecuta face_engine.py y retorna el resultado."""
    cmd = [python_exe, engine_path, action]
    if username:
        cmd.append(username)
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE_DIR)
    try:
        return json.loads(proc.stdout.strip())
    except:
        return {"error": proc.stdout + proc.stderr}

# --- Listar perfiles ---
result = run_engine("list")
profiles = result.get("profiles", [])
print(f"👥 Perfiles registrados ({len(profiles)}): {profiles}")

# --- Descomenta para ver info o eliminar ---
# info = run_engine("info", "andi")
# print(json.dumps(info, indent=2, ensure_ascii=False))

# result = run_engine("delete", "andi")
# print(result)

👥 Perfiles registrados (3): ['Anderson', 'Andi', 'ron']


## 🖥️ Bloque 12: Interfaz Gráfica Embebida (Tkinter)
Lanza una **ventana Tkinter premium** directamente desde el notebook con interfaz moderna:

- 🎨 **Diseño oscuro profesional** con paleta violeta + verde + negro azulado
- ✏️ **Campo de usuario** con placeholder y glow al hacer focus
- 📷 **Registrar** / 🔐 **Login** con botones hover interactivos
- 👥 **Lista de perfiles** (clic para seleccionar)
- ℹ️ **Info** / 🗑️ **Eliminar** con mini-botones hover
- 📋 **Log de actividad** con colores por tipo (éxito, error, info)
- 🟢 **Indicador pulsante** de estado (verde=listo, naranja=procesando)
- 🌈 **Barra gradiente** decorativa violeta→rosa

> 💡 **Multiusuario**: Cada usuario tiene su propio perfil `.npz`. Registra múltiples personas.

In [1]:
# =============================================
# 🖥️ INTERFAZ GRÁFICA PREMIUM (Tkinter embebida)
# =============================================
# Se ejecuta en un thread separado para no bloquear el notebook.
# Cierra la ventana cuando termines.

import tkinter as tk
from tkinter import messagebox
import subprocess, json, shutil, threading, math
from datetime import datetime as _dt

# ── Config ──
_BASE = os.path.dirname(os.path.abspath("__file__"))
_PROFILES = os.path.join(_BASE, "profiles")
_ENGINE = os.path.join(_BASE, "face_engine.py")
_PY = shutil.which("python", path=os.path.expanduser("~/anaconda3/envs/insightface/bin")) or sys.executable

# ── Paleta ──
_C = {
    "bg": "#0f0f1a", "surface": "#181830", "surface2": "#1e1e3a",
    "border": "#2a2a50", "accent": "#6c63ff", "accent_h": "#8b83ff",
    "accent_d": "#4a42cc", "danger": "#ff4757", "danger_h": "#ff6b7a",
    "success": "#2ed573", "success_d": "#1ea85a", "warning": "#ffa502",
    "text": "#e8e8f0", "text2": "#9090b0", "text3": "#606080",
    "input_bg": "#12122a", "log_bg": "#0a0a18", "log_fg": "#70e090",
    "grad1": "#6c63ff", "grad2": "#e94580",
}

# ══════════════════════════════════════════════
# Widgets personalizados
# ══════════════════════════════════════════════

class _GradientBar(tk.Canvas):
    def __init__(self, parent, height=3, c1=None, c2=None, **kw):
        self.c1, self.c2 = c1 or _C["grad1"], c2 or _C["grad2"]
        bg = kw.pop("bg", None) or parent.cget("bg")
        super().__init__(parent, height=height, highlightthickness=0, bg=bg, **kw)
        self.bind("<Configure>", self._draw)
    def _draw(self, e=None):
        self.delete("all")
        w, h = self.winfo_width(), self.winfo_height()
        if w < 2: return
        r1,g1,b1 = self._h(self.c1); r2,g2,b2 = self._h(self.c2)
        for i in range(w):
            t = i/max(w-1,1)
            c = f"#{int(r1+(r2-r1)*t):02x}{int(g1+(g2-g1)*t):02x}{int(b1+(b2-b1)*t):02x}"
            self.create_line(i, 0, i, h, fill=c)
    @staticmethod
    def _h(x):
        x = x.lstrip("#"); return tuple(int(x[i:i+2],16) for i in (0,2,4))

class _PulsingDot(tk.Canvas):
    def __init__(self, parent, size=10, color=None, **kw):
        bg = kw.pop("bg", None) or parent.cget("bg")
        super().__init__(parent, width=size+10, height=size+10, highlightthickness=0, bg=bg, **kw)
        self._s, self._c, self._run, self._step = size, color or _C["success"], False, 0
        self._draw()
    def _draw(self):
        self.delete("all"); m=(self._s+10)//2; r=self._s//2
        self.create_oval(m-r,m-r,m+r,m+r,fill=self._c,outline="")
    def start_pulse(self, c=None):
        if c: self._c = c
        self._run, self._step = True, 0; self._tick()
    def stop_pulse(self, c=None):
        self._run = False
        if c: self._c = c
        self._draw()
    def _tick(self):
        if not self._run: return
        self.delete("all"); m=(self._s+10)//2; r=self._s//2
        self._step = (self._step+1) % 24
        s = 1.0 + 0.6*math.sin(self._step*math.pi/12); pr=int(r*s)
        self.create_oval(m-pr,m-pr,m+pr,m+pr,fill="",outline=self._c,width=1)
        self.create_oval(m-r,m-r,m+r,m+r,fill=self._c,outline="")
        self.after(70, self._tick)

class _Btn(tk.Frame):
    def __init__(self, parent, text="", icon="", command=None,
                 bg_color=None, hover_color=None, press_color=None,
                 fg="#ffffff", width=18, font_size=12, font_weight="bold"):
        super().__init__(parent, bg=parent.cget("bg"))
        self._bg = bg_color or _C["accent"]
        self._hover = hover_color or _C["accent_h"]
        self._press = press_color or _C["accent_d"]
        self._cmd, self._ok = command, True
        display = f"{icon}  {text}" if icon else text
        self._glow = tk.Frame(self, bg=self._bg, padx=1, pady=1)
        self._glow.pack(fill=tk.BOTH, expand=True)
        self._lbl = tk.Label(self._glow, text=display,
            font=("Segoe UI", font_size, font_weight),
            fg=fg, bg=self._bg, cursor="hand2", width=width, pady=8, padx=12)
        self._lbl.pack(fill=tk.BOTH, expand=True, padx=2, pady=2)
        for w in (self._glow, self._lbl):
            w.bind("<Enter>", lambda e: self._set(self._hover) if self._ok else None)
            w.bind("<Leave>", lambda e: self._set(self._bg) if self._ok else None)
            w.bind("<ButtonPress-1>", lambda e: self._set(self._press) if self._ok else None)
            w.bind("<ButtonRelease-1>", self._click)
    def _set(self, c):
        self._lbl.config(bg=c); self._glow.config(bg=c)
    def _click(self, e):
        if self._ok:
            self._set(self._hover)
            if self._cmd: self._cmd()
    def set_enabled(self, v):
        self._ok = v; c = self._bg if v else _C["text3"]
        self._lbl.config(bg=c, cursor="hand2" if v else "arrow"); self._glow.config(bg=c)

class _Mini(tk.Label):
    def __init__(self, parent, text="", command=None,
                 bg_color=None, hover_color=None, fg_color=None, **kw):
        self._bg = bg_color or _C["surface2"]
        self._hover = hover_color or _C["border"]
        self._fg = fg_color or _C["text2"]
        self._cmd = command
        super().__init__(parent, text=text, font=("Segoe UI", 8),
            bg=self._bg, fg=self._fg, cursor="hand2", padx=8, pady=3, **kw)
        self.bind("<Enter>", lambda e: self.config(bg=self._hover, fg=_C["text"]))
        self.bind("<Leave>", lambda e: self.config(bg=self._bg, fg=self._fg))
        self.bind("<ButtonRelease-1>", lambda e: self._cmd() if self._cmd else None)

# ══════════════════════════════════════════════
# App principal
# ══════════════════════════════════════════════

class _FaceApp:
    def __init__(self, root):
        self.root, self.is_running = root, False
        root.title("  Autenticación Facial · InsightFace")
        root.configure(bg=_C["bg"])
        self._build(); self._refresh()

    def _build(self):
        w = tk.Frame(self.root, bg=_C["bg"])
        w.pack(fill=tk.BOTH, expand=True)

        # ── Header ──
        hdr = tk.Frame(w, bg=_C["bg"]); hdr.pack(fill=tk.X, padx=30, pady=(26,0))
        logo = tk.Canvas(hdr, width=52, height=52, highlightthickness=0, bg=_C["bg"])
        logo.pack(side=tk.LEFT, padx=(0,16))
        logo.create_oval(2,2,50,50,outline=_C["accent"],width=2.5,fill="")
        logo.create_oval(8,8,44,44,fill=_C["accent"],outline="")
        logo.create_oval(19,12,33,27,fill="#ffffff",outline="")
        logo.create_arc(13,24,39,48,start=0,extent=180,fill="#ffffff",outline="")
        txt = tk.Frame(hdr, bg=_C["bg"]); txt.pack(side=tk.LEFT, fill=tk.X, expand=True)
        tk.Label(txt, text="Autenticación Facial", font=("Segoe UI",22,"bold"),
                 fg=_C["text"], bg=_C["bg"]).pack(anchor=tk.W)
        tk.Label(txt, text="InsightFace ArcFace  ·  Multiusuario  ·  Offline",
                 font=("Segoe UI",9), fg=_C["text3"], bg=_C["bg"]).pack(anchor=tk.W, pady=(2,0))
        self.dot = _PulsingDot(hdr, size=10, color=_C["success"])
        self.dot.pack(side=tk.RIGHT, pady=12)
        _GradientBar(w, height=2).pack(fill=tk.X, padx=30, pady=(16,22))

        # ── Usuario ──
        s1 = tk.Frame(w, bg=_C["bg"]); s1.pack(fill=tk.X, padx=30)
        tk.Label(s1, text="USUARIO", font=("Segoe UI",9,"bold"),
                 fg=_C["accent"], bg=_C["bg"]).pack(anchor=tk.W, pady=(0,8))
        self._ib = tk.Frame(s1, bg=_C["border"], padx=2, pady=2); self._ib.pack(fill=tk.X, pady=(0,2))
        ii = tk.Frame(self._ib, bg=_C["input_bg"]); ii.pack(fill=tk.X)
        tk.Label(ii, text="  👤 ", font=("Segoe UI",14),
                 bg=_C["input_bg"], fg=_C["text3"]).pack(side=tk.LEFT, padx=(8,0))
        self.uv = tk.StringVar()
        self.ent = tk.Entry(ii, textvariable=self.uv, font=("Segoe UI",14),
            bg=_C["input_bg"], fg=_C["text"], insertbackground=_C["accent"],
            relief=tk.FLAT, highlightthickness=0, bd=0)
        self.ent.pack(fill=tk.X, side=tk.LEFT, expand=True, padx=(6,14), pady=11, ipady=2)
        self.ent.focus_set()
        self._ph = True; self._set_ph()
        self.ent.bind("<FocusIn>", self._fi); self.ent.bind("<FocusOut>", self._fo)
        self.ent.bind("<Return>", lambda e: self._login())
        tk.Label(s1, text="Escribe un nombre  ·  Enter = login rápido",
                 font=("Segoe UI",8), fg=_C["text3"], bg=_C["bg"]).pack(anchor=tk.W, pady=(4,0))

        # ── Acciones ──
        s2 = tk.Frame(w, bg=_C["bg"]); s2.pack(fill=tk.X, padx=30, pady=(18,0))
        tk.Label(s2, text="ACCIONES", font=("Segoe UI",9,"bold"),
                 fg=_C["accent"], bg=_C["bg"]).pack(anchor=tk.W, pady=(0,8))
        row = tk.Frame(s2, bg=_C["bg"]); row.pack(fill=tk.X)
        self.br = _Btn(row, text="Registrar Rostro", icon="📷", command=self._register,
            bg_color=_C["accent"], hover_color=_C["accent_h"], press_color=_C["accent_d"], width=18)
        self.br.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0,5))
        self.bl = _Btn(row, text="Iniciar Sesión", icon="🔐", command=self._login,
            bg_color=_C["success_d"], hover_color=_C["success"], press_color="#178a47", width=18)
        self.bl.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(5,0))

        # ── Perfiles ──
        s3 = tk.Frame(w, bg=_C["bg"]); s3.pack(fill=tk.X, padx=30, pady=(20,0))
        ph = tk.Frame(s3, bg=_C["bg"]); ph.pack(fill=tk.X, pady=(0,8))
        tk.Label(ph, text="PERFILES REGISTRADOS", font=("Segoe UI",9,"bold"),
                 fg=_C["accent"], bg=_C["bg"]).pack(side=tk.LEFT)
        ut = tk.Frame(ph, bg=_C["bg"]); ut.pack(side=tk.RIGHT)
        _Mini(ut, text=" ⟳ Refresh ", command=self._refresh).pack(side=tk.LEFT, padx=2)
        _Mini(ut, text=" ℹ Info ", command=self._info).pack(side=tk.LEFT, padx=2)
        _Mini(ut, text=" ✕ Eliminar ", command=self._delete,
              bg_color="#2a1525", hover_color="#3d1a30", fg_color=_C["danger"]).pack(side=tk.LEFT, padx=2)
        lb_b = tk.Frame(s3, bg=_C["border"], padx=1, pady=1); lb_b.pack(fill=tk.X)
        lb_i = tk.Frame(lb_b, bg=_C["surface"]); lb_i.pack(fill=tk.X)
        self.lb = tk.Listbox(lb_i, height=4, font=("Segoe UI",11), bg=_C["surface"],
            fg=_C["text2"], selectbackground=_C["accent"], selectforeground="#ffffff",
            relief=tk.FLAT, highlightthickness=0, activestyle="none", bd=0, selectborderwidth=0)
        self.lb.pack(fill=tk.X, side=tk.LEFT, expand=True, padx=10, pady=8)
        self.lb.bind("<<ListboxSelect>>", self._sel)
        sb = tk.Scrollbar(lb_i, command=self.lb.yview, troughcolor=_C["surface"], bg=_C["border"])
        sb.pack(side=tk.RIGHT, fill=tk.Y, padx=(0,4), pady=8)
        self.lb.config(yscrollcommand=sb.set)

        # ── Log ──
        s4 = tk.Frame(w, bg=_C["bg"]); s4.pack(fill=tk.BOTH, expand=True, padx=30, pady=(18,0))
        tk.Label(s4, text="ACTIVIDAD", font=("Segoe UI",9,"bold"),
                 fg=_C["accent"], bg=_C["bg"]).pack(anchor=tk.W, pady=(0,6))
        lb2 = tk.Frame(s4, bg=_C["border"], padx=1, pady=1); lb2.pack(fill=tk.BOTH, expand=True)
        li2 = tk.Frame(lb2, bg=_C["log_bg"]); li2.pack(fill=tk.BOTH, expand=True)
        self.log = tk.Text(li2, height=7, font=("JetBrains Mono",9), bg=_C["log_bg"],
            fg=_C["log_fg"], insertbackground=_C["accent"], relief=tk.FLAT, wrap=tk.WORD,
            state=tk.DISABLED, bd=0, padx=14, pady=10, spacing1=3)
        self.log.pack(fill=tk.BOTH, expand=True, side=tk.LEFT)
        ls = tk.Scrollbar(li2, command=self.log.yview, troughcolor=_C["log_bg"], bg=_C["border"])
        ls.pack(side=tk.RIGHT, fill=tk.Y)
        self.log.config(yscrollcommand=ls.set)
        for t,c in [("ok",_C["success"]),("err",_C["danger"]),("warn",_C["warning"]),
                     ("info",_C["accent"]),("dim",_C["text3"])]:
            self.log.tag_configure(t, foreground=c)

        # ── Status bar ──
        bar = tk.Frame(self.root, bg=_C["surface"], height=30)
        bar.pack(fill=tk.X, side=tk.BOTTOM); bar.pack_propagate(False)
        _GradientBar(bar, height=1, bg=_C["surface"]).pack(fill=tk.X, side=tk.TOP)
        self.sv = tk.StringVar(value="  ● Listo")
        tk.Label(bar, textvariable=self.sv, font=("Segoe UI",8),
                 bg=_C["surface"], fg=_C["text3"], anchor=tk.W).pack(fill=tk.X, padx=14, pady=3)

        self._wl("Sistema biométrico iniciado", "info")
        self._wl("Motor: InsightFace ArcFace · buffalo_l", "dim")

    # ── Engine ──
    def _eng(self, action, user=None):
        cmd = [_PY, _ENGINE, action]
        if user: cmd.append(user)
        p = subprocess.run(cmd, capture_output=True, text=True, cwd=_BASE)
        return p.stdout, p.stderr

    def _eng_async(self, action, user, cb):
        def work():
            self.is_running = True
            self.root.after(0, lambda: self._btns(False))
            self.root.after(0, lambda: self.dot.start_pulse(_C["warning"]))
            self.root.after(0, lambda: self.sv.set(f"  ◉ {action}... (cámara abierta)"))
            out, err = self._eng(action, user)
            res, lines = None, []
            for ln in out.strip().split('\n'):
                if ln.startswith("__RESULT_JSON__:"):
                    try: res = json.loads(ln.replace("__RESULT_JSON__:",""))
                    except: pass
                elif ln.strip(): lines.append(ln)
            self.is_running = False
            self.root.after(0, lambda: self._btns(True))
            self.root.after(0, lambda: self.dot.stop_pulse(_C["success"]))
            self.root.after(0, lambda: cb(res, lines, err))
        threading.Thread(target=work, daemon=True).start()

    # ── Actions ──
    def _register(self):
        u = self._gu()
        if not u:
            messagebox.showwarning("Requerido", "Ingresa nombre de usuario."); self.ent.focus_set(); return
        if not u.replace("_","").isalnum():
            messagebox.showwarning("Inválido", "Solo letras, números y _"); return
        if self.is_running: return
        self._wl(f"Iniciando registro para '{u}'...", "info")
        self._eng_async("register", u, self._cb_reg)

    def _cb_reg(self, res, lines, err):
        for ln in lines: self._wl(f"  {ln}", "dim")
        if res and res.get("success"):
            u,c,d,t = res['username'],res['num_captures'],res['embedding_dim'],res['processing_time_s']
            self._wl(f"✓ Registrado: {u}  ({c} capturas, {d}D, {t}s)", "ok")
            self.sv.set(f"  ● Perfil '{u}' registrado")
            messagebox.showinfo("Registro Exitoso", f"Perfil '{u}' creado.\n\n  Capturas: {c}\n  Embedding: {d}D\n  Tiempo: {t}s")
        else:
            msg = res.get("message","Error") if res else "Sin respuesta"
            self._wl(f"✗ Fallido: {msg}", "err"); self.sv.set("  ● Error")
            messagebox.showerror("Fallido", msg)
        self._refresh()

    def _login(self):
        u = self._gu()
        if not u:
            messagebox.showwarning("Requerido", "Ingresa nombre."); self.ent.focus_set(); return
        if self.is_running: return
        if u not in self._gp():
            messagebox.showwarning("No encontrado", f"No existe '{u}'. Regístralo."); return
        self._wl(f"Verificando '{u}'...", "info")
        self._eng_async("login", u, self._cb_login)

    def _cb_login(self, res, lines, err):
        for ln in lines: self._wl(f"  {ln}", "dim")
        if res and res.get("match"):
            sc,conf,ms = res.get("score",0),res.get("confidence_level","N/A"),res.get("processing_time_ms",0)
            self._wl(f"✓ ACCESO CONCEDIDO  score={sc:.3f}  ({conf})", "ok")
            self.sv.set(f"  ● Login OK · {sc:.3f}")
            messagebox.showinfo("Concedido", f"Verificado.\n\n  Score: {sc:.3f}\n  Confianza: {conf}\n  Tiempo: {ms}ms")
        else:
            sc = res.get("score",0) if res else 0
            msg = res.get("message","No match") if res else "Sin respuesta"
            self._wl(f"✗ DENEGADO  score={sc:.3f}", "err"); self.sv.set(f"  ● Fallido · {sc:.3f}")
            messagebox.showerror("Denegado", f"No verificado.\n\n  Score: {sc:.3f}\n  {msg}")

    def _info(self):
        u = self._gs()
        if not u: messagebox.showinfo("Info","Selecciona un perfil."); return
        out,_ = self._eng("info", u)
        try:
            i = json.loads(out.strip())
            if "error" in i: messagebox.showerror("Error",i["error"]); return
            ll = i.get('last_login','Nunca'); ll = ll[:19] if ll != 'Nunca' else ll
            messagebox.showinfo(f"Perfil: {u}",
                f"  Usuario:        {i['username']}\n  Creado:          {i['created'][:19]}\n"
                f"  Capturas:      {i['num_captures']}\n  Logins:           {i['login_count']}\n"
                f"  Último login:  {ll}\n  Embedding:   {i['embedding_shape']}  {i['embedding_dtype']}")
            self._wl(f"ℹ {u}: {i['num_captures']} capturas, {i['login_count']} logins", "info")
        except Exception as e: messagebox.showerror("Error",str(e))

    def _delete(self):
        u = self._gs()
        if not u: messagebox.showinfo("Eliminar","Selecciona un perfil."); return
        if not messagebox.askyesno("Confirmar",f"¿Eliminar '{u}'?\n\nNo se puede deshacer."): return
        out,_ = self._eng("delete", u)
        try:
            r = json.loads(out.strip())
            if r.get("deleted"):
                self._wl(f"✓ '{u}' eliminado","warn"); messagebox.showinfo("OK",f"'{u}' eliminado.")
            else: messagebox.showerror("Error",r.get("message","Error"))
        except Exception as e: messagebox.showerror("Error",str(e))
        self._refresh()

    def _sel(self, e):
        u = self._gs()
        if u: self._ph=False; self.ent.config(fg=_C["text"]); self.uv.set(u)

    # ── Helpers ──
    def _set_ph(self):
        if not self.uv.get().strip() or self._ph:
            self._ph=True; self.ent.config(fg=_C["text3"]); self.uv.set("nombre de usuario...")
    def _fi(self, e):
        self._ib.config(bg=_C["accent"])
        if self._ph: self.uv.set(""); self.ent.config(fg=_C["text"]); self._ph=False
    def _fo(self, e):
        self._ib.config(bg=_C["border"])
        if not self.uv.get().strip(): self._set_ph()
    def _gu(self):
        v = self.uv.get().strip()
        return "" if self._ph or v=="nombre de usuario..." else v
    def _gp(self):
        try:
            out,_ = self._eng("list"); return json.loads(out.strip()).get("profiles",[])
        except: return []
    def _refresh(self):
        ps = self._gp(); self.lb.delete(0, tk.END)
        if ps:
            for p in sorted(ps): self.lb.insert(tk.END, f"    ●  {p}")
            self.sv.set(f"  ● {len(ps)} perfil(es)")
        else:
            self.lb.insert(tk.END, "     Sin perfiles registrados")
            self.sv.set("  ○ Sin perfiles — Registra uno")
    def _gs(self):
        sel = self.lb.curselection()
        if not sel: return None
        t = self.lb.get(sel[0]).strip()
        return t.replace("●","").strip() if "●" in t else None
    def _btns(self, ok): self.br.set_enabled(ok); self.bl.set_enabled(ok)
    def _wl(self, msg, tag=None):
        ts = _dt.now().strftime("%H:%M:%S")
        self.log.config(state=tk.NORMAL)
        self.log.insert(tk.END, f"[{ts}]  ", "dim")
        self.log.insert(tk.END, f"{msg}\n", tag)
        self.log.see(tk.END); self.log.config(state=tk.DISABLED)

# ══════════════════════════════════════════════
# Lanzar en thread separado
# ══════════════════════════════════════════════

def _launch_gui():
    root = tk.Tk()
    root.update_idletasks()
    w, h = 660, 800
    x = (root.winfo_screenwidth() - w) // 2
    y = (root.winfo_screenheight() - h) // 2
    root.geometry(f"{w}x{h}+{x}+{y}")
    root.minsize(600, 720)
    _FaceApp(root)
    root.mainloop()

gui_thread = threading.Thread(target=_launch_gui, daemon=True)
gui_thread.start()

print("🖥️ Interfaz gráfica lanzada en ventana separada")
print("📺 Panel de autenticación facial con diseño premium")
print("👥 Multiusuario: registra y loguea múltiples personas")
print("💡 Cierra la ventana Tkinter cuando termines")

NameError: name 'os' is not defined